# Módulo 6 · Clase 5 — Inferencia estadística

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Población vs muestra |
| 2 | Distribución muestral y error estándar |
| 3 | Teorema Central del Límite |
| 4 | Intervalos de confianza |
| 5 | Test de hipótesis (t-test) |
| 6 | Errores tipo I y II |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print("Librerías listas")

---
## Motivación

```
¿Cuánto tarda en promedio mi modelo en hacer una predicción?
¿Este nuevo modelo es significativamente mejor que el anterior?
¿Es real la diferencia que veo o pudo ocurrir por azar?

La estadística inferencial responde estas preguntas a partir de datos limitados.

Nunca podemos observar TODO → inferimos sobre la población
usando lo que observamos en una muestra.
```

---
## 1. Población vs muestra

In [ ]:
# Simulamos la "población real" (normalmente inaccesible)
np.random.seed(42)
POBLACION_TIEMPOS = np.random.gamma(shape=2, scale=50, size=100_000)  # ms de latencia

mu_real    = POBLACION_TIEMPOS.mean()
sigma_real = POBLACION_TIEMPOS.std(ddof=0)

print("=== Parámetros REALES de la población ===")
print(f"μ (media real)  = {mu_real:.4f} ms")
print(f"σ (desv. real)  = {sigma_real:.4f} ms")
print(f"Tamaño población: {len(POBLACION_TIEMPOS):,}")

# Tomamos muestras de tamaño n
print("\n=== Estimaciones desde muestras ===")
print(f"{'n':>6} {'media_muestra':>15} {'error':>10} {'std_muestra':>13}")

for n in [5, 10, 30, 100, 1000]:
    muestra   = np.random.choice(POBLACION_TIEMPOS, size=n, replace=False)
    media_m   = muestra.mean()
    std_m     = muestra.std(ddof=1)
    error     = abs(media_m - mu_real)
    print(f"{n:>6} {media_m:>15.4f} {error:>10.4f} {std_m:>13.4f}")

print(f"\n→ A mayor n, las estimaciones se acercan más a los parámetros reales")

---
## 2. Distribución muestral

```
Si tomamos muchas muestras y calculamos la media de cada una,
esas medias forman la "distribución muestral de la media".

Media de la distribución muestral = μ (media poblacional)
Desv. estándar de la distribución muestral = Error Estándar = σ / √n

A mayor n → menor error estándar → estimaciones más precisas
```

In [ ]:
n_muestras = 1000
fig, axes  = plt.subplots(1, 3, figsize=(14, 4))

for ax, n in zip(axes, [5, 30, 100]):
    medias = [np.random.choice(POBLACION_TIEMPOS, size=n).mean() for _ in range(n_muestras)]
    medias = np.array(medias)

    error_std_teorico = sigma_real / np.sqrt(n)
    error_std_empir   = medias.std()

    ax.hist(medias, bins=40, density=True, alpha=0.7, color='steelblue', edgecolor='white')
    # Curva normal teórica
    x_plot = np.linspace(medias.min(), medias.max(), 200)
    ax.plot(x_plot, stats.norm.pdf(x_plot, mu_real, error_std_teorico),
            'r-', lw=2, label='Normal teórica')
    ax.axvline(mu_real, color='orange', ls='--', lw=2, label=f'μ={mu_real:.1f}')
    ax.set_title(f'n={n}\nEE teórico={error_std_teorico:.2f}, empírico={error_std_empir:.2f}')
    ax.legend(fontsize=7)
    ax.set_xlabel('Media muestral (ms)')

plt.suptitle('Distribución muestral de la media (1000 muestras)', fontsize=11)
plt.tight_layout(); plt.show()

print("Observaciones:")
print("  1. Las medias muestrales se distribuyen alrededor de μ real")
print("  2. A mayor n, más estrecha es la distribución (más precisión)")
print("  3. Aunque la población sea asimétrica (gamma), las medias son normales")

---
## 3. Teorema Central del Límite (TCL)

```
Sin importar la forma de la distribución original,
si n es suficientemente grande (≥30),
la distribución muestral de la media se aproxima a una normal.

→ Es lo que hace posible la inferencia estadística en la práctica.
→ En IA: justifica asumir normalidad en residuos, errores de predicción, etc.
```

In [ ]:
# Demostrar TCL con 3 distribuciones muy distintas
np.random.seed(42)
distribuciones = [
    ('Uniforme', lambda n: np.random.uniform(0, 1, n)),
    ('Exponencial', lambda n: np.random.exponential(1, n)),
    ('Bernoulli p=0.1', lambda n: np.random.binomial(1, 0.1, n)),
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for row, (nombre, generador) in enumerate(distribuciones):
    # Población
    pob = generador(10000)
    axes[row, 0].hist(pob, bins=40, density=True, alpha=0.7, color='coral')
    axes[row, 0].set_title(f'{nombre}\n(distribución original)')

    # Distribuciones muestrales para n=2, 10, 30
    for col, n in enumerate([2, 10, 30]):
        medias = [generador(n).mean() for _ in range(2000)]
        medias = np.array(medias)

        axes[row, col+1].hist(medias, bins=40, density=True, alpha=0.7,
                              color='steelblue', edgecolor='white')

        # Overlay normal teórica
        mu_t  = medias.mean()
        std_t = medias.std()
        x_n   = np.linspace(medias.min(), medias.max(), 200)
        axes[row, col+1].plot(x_n, stats.norm.pdf(x_n, mu_t, std_t), 'r-', lw=2)
        skew_m = stats.skew(medias)
        axes[row, col+1].set_title(f'n={n}\nskewness={skew_m:.3f}')

plt.suptitle('Teorema Central del Límite', fontsize=13)
plt.tight_layout(); plt.show()

print("Para n=30, la distribución de las medias es casi normal en los tres casos.")

---
## 4. Intervalos de confianza

```
"Con 95% de confianza, la media real está entre [a, b]"

IC al 95% = x̄ ± t* × (s/√n)

  x̄  = media muestral
  s   = desviación estándar muestral
  n   = tamaño de muestra
  t*  = valor crítico de la t de Student (≈1.96 para n grande, 95%)

Interpretación CORRECTA:
  Si repitieras el experimento 100 veces, aproximadamente 95 de los
  intervalos contendrían la media real.

No significa: "hay 95% de probabilidad de que μ esté en [a, b]"
```

In [ ]:
def intervalo_confianza(datos, confianza=0.95):
    n    = len(datos)
    mean = np.mean(datos)
    se   = stats.sem(datos)  # error estándar = s / sqrt(n)
    ic   = stats.t.interval(confianza, df=n-1, loc=mean, scale=se)
    return mean, ic[0], ic[1], se

# Una muestra
np.random.seed(42)
muestra = np.random.choice(POBLACION_TIEMPOS, size=30)
mean, ic_inf, ic_sup, se = intervalo_confianza(muestra)

print(f"Muestra n=30:")
print(f"  Media muestral:    {mean:.4f} ms")
print(f"  Error estándar:    {se:.4f} ms")
print(f"  IC 95%:            [{ic_inf:.4f}, {ic_sup:.4f}] ms")
print(f"  μ real:            {mu_real:.4f} ms → {'Dentro ✓' if ic_inf <= mu_real <= ic_sup else 'Fuera ✗'}")

# Efecto del nivel de confianza
print("\nIC para la misma muestra, distintos niveles:")
for conf in [0.80, 0.90, 0.95, 0.99]:
    m, a, b, _ = intervalo_confianza(muestra, conf)
    print(f"  {int(conf*100)}%: [{a:.2f}, {b:.2f}]  (ancho={b-a:.2f})")

# Visualizar 100 ICs
np.random.seed(0)
n_experimentos = 100
exitos = 0
fig, ax = plt.subplots(figsize=(12, 6))

for i in range(n_experimentos):
    m_i = np.random.choice(POBLACION_TIEMPOS, size=30)
    mean_i, a_i, b_i, _ = intervalo_confianza(m_i)
    dentro = a_i <= mu_real <= b_i
    if dentro: exitos += 1
    color = 'steelblue' if dentro else 'red'
    ax.plot([a_i, b_i], [i, i], color=color, alpha=0.5, lw=1)
    ax.plot(mean_i, i, 'o', color=color, ms=2)

ax.axvline(mu_real, color='orange', ls='--', lw=2, label=f'μ real = {mu_real:.1f}')
ax.set_xlabel('Latencia (ms)')
ax.set_title(f'100 intervalos de confianza al 95% — Contienen μ: {exitos}/100')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\n{exitos} de 100 ICs contienen la media real (esperado ≈ 95)")

---
## 5. Test de hipótesis

```
Queremos saber si una diferencia observada es real o fruto del azar.

H₀ (hipótesis nula):     no hay diferencia / efecto
H₁ (hipótesis alternativa): sí hay diferencia / efecto

p-valor: probabilidad de obtener este resultado (o más extremo)
         si H₀ fuera verdad.

  p < α (ej. 0.05)  → rechazamos H₀ → diferencia significativa
  p ≥ α             → no podemos rechazar H₀

¡OJO! "No rechazar H₀" ≠ "H₀ es verdad"
       "Significativo" ≠ "importante" o "grande"
```

In [ ]:
# Escenario: comparar latencia de dos versiones del modelo
np.random.seed(42)
version_A = np.random.normal(100, 15, 30)  # ms
version_B = np.random.normal(95, 18, 30)   # ms (¿es B realmente más rápida?)

print(f"Versión A: media={version_A.mean():.2f} ms, std={version_A.std():.2f}")
print(f"Versión B: media={version_B.mean():.2f} ms, std={version_B.std():.2f}")
print(f"Diferencia observada: {version_A.mean() - version_B.mean():.2f} ms")

# t-test independiente (Welch)
t_stat, p_valor = stats.ttest_ind(version_A, version_B, equal_var=False)
alpha = 0.05

print(f"\nt-estadístico: {t_stat:.4f}")
print(f"p-valor:       {p_valor:.4f}")
print(f"Alpha:         {alpha}")
print(f"Decisión: {'Rechazamos H₀ — diferencia significativa' if p_valor < alpha else 'No rechazamos H₀ — diferencia no significativa'}")

# t-test de una muestra (¿la media es diferente de un valor?)
print("\n=== ¿La latencia de B supera el umbral de 90ms? ===")
t2, p2 = stats.ttest_1samp(version_B, popmean=90)
print(f"H₀: μ_B = 90ms")
print(f"t={t2:.4f}, p={p2:.4f}")
print(f"Decisión: {'Sí supera significativamente 90ms' if p2 < alpha else 'No supera significativamente 90ms'}")

# t-test pareado (mismo modelo, antes y después de optimización)
print("\n=== Test pareado: antes y después de optimización ===")
antes = np.random.normal(120, 10, 20)
despues = antes - np.random.normal(8, 5, 20)  # mejora media de 8ms
t3, p3 = stats.ttest_rel(antes, despues)
print(f"Reducción media: {(antes - despues).mean():.2f} ms")
print(f"t={t3:.4f}, p={p3:.4f}")
print(f"La optimización {'es' if p3 < alpha else 'NO es'} estadísticamente significativa")

---
## 6. Errores tipo I y II

```
                    H₀ verdad         H₀ falsa
Rechazar H₀    ┃ Error Tipo I (α) ┃ Correcto (Poder) ┃
No rechazar H₀ ┃ Correcto         ┃ Error Tipo II (β)┃

Tipo I  (Falso Positivo): "Detectamos algo que no existe"
         Probabilidad = α (nivel de significancia)

Tipo II (Falso Negativo): "No detectamos algo que existe"
         Probabilidad = β

Poder estadístico = 1 - β = probabilidad de detectar efecto real
  Aumenta con: mayor n, mayor efecto, mayor α, menor variabilidad
```

In [ ]:
# Simular tipo I y tipo II
np.random.seed(42)
n_sim    = 2000
alpha    = 0.05

# Error tipo I: H₀ verdad (μ_A = μ_B = 100)
falsas_positivas = sum(
    1 for _ in range(n_sim)
    if stats.ttest_ind(np.random.normal(100, 15, 30),
                       np.random.normal(100, 15, 30))[1] < alpha
)
print(f"Errores Tipo I (esperado ~{alpha*100:.0f}%): {falsas_positivas/n_sim*100:.1f}%")

# Error tipo II: H₀ falsa (μ_B = 93, diferencia real de 7ms)
for n in [10, 30, 100]:
    detectados = sum(
        1 for _ in range(n_sim)
        if stats.ttest_ind(np.random.normal(100, 15, n),
                           np.random.normal(93, 15, n))[1] < alpha
    )
    poder = detectados / n_sim
    tipo2 = 1 - poder
    print(f"n={n:3d}: Poder={poder:.3f} ({poder*100:.1f}%), Error Tipo II={tipo2:.3f}")

# Visualización
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(80, 130, 400)

dist_H0 = stats.norm(100, 15/np.sqrt(30))
dist_H1 = stats.norm(93, 15/np.sqrt(30))
t_critico_val = dist_H0.ppf(alpha / 2)  # test bilateral
t_critico_sup = dist_H0.ppf(1 - alpha / 2)

ax.plot(x, dist_H0.pdf(x), 'b-', lw=2, label='H₀: μ=100 (sin diferencia)')
ax.plot(x, dist_H1.pdf(x), 'r-', lw=2, label='H₁: μ=93 (diferencia real)')

# Zona de rechazo (Tipo I)
x_alpha = np.linspace(80, t_critico_val, 100)
ax.fill_between(x_alpha, dist_H0.pdf(x_alpha), alpha=0.3, color='blue', label='Error Tipo I (α)')
x_alpha2 = np.linspace(t_critico_sup, 130, 100)
ax.fill_between(x_alpha2, dist_H0.pdf(x_alpha2), alpha=0.3, color='blue')

# Zona Tipo II (β)
x_beta = np.linspace(t_critico_val, t_critico_sup, 200)
ax.fill_between(x_beta, dist_H1.pdf(x_beta), alpha=0.3, color='red', label='Error Tipo II (β)')

ax.axvline(t_critico_val, color='purple', ls='--', alpha=0.5)
ax.axvline(t_critico_sup, color='purple', ls='--', alpha=0.5, label='Valor crítico (α=0.05)')
ax.set_xlabel('Media muestral (ms)'); ax.set_ylabel('Densidad')
ax.set_title('Errores Tipo I y Tipo II')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Práctica

### Ejercicio 1 — Intervalo de confianza para métricas de modelo

**Enunciado:** Evalúas tu modelo en 40 ejemplos y obtienes estas accuracies por batch:

```python
np.random.seed(5)
accuracies = np.random.beta(8, 2, 40)  # ~media 0.80
```

1. Media y desviación estándar
2. IC al 90%, 95% y 99%
3. ¿Puedes afirmar con 95% de confianza que el modelo supera el 75% de accuracy?
4. ¿Cuántos ejemplos necesitarías para que el IC 95% tenga un ancho máximo de 0.05?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
from scipy import stats

np.random.seed(5)
accuracies = np.random.beta(8, 2, 40)

# 1.
media = accuracies.mean()
std   = accuracies.std(ddof=1)
print(f"Media: {media:.4f}  |  Std: {std:.4f}")

# 2.
for conf in [0.90, 0.95, 0.99]:
    ic = stats.t.interval(conf, df=len(accuracies)-1,
                          loc=media, scale=stats.sem(accuracies))
    ancho = ic[1] - ic[0]
    print(f"IC {int(conf*100)}%: [{ic[0]:.4f}, {ic[1]:.4f}]  ancho={ancho:.4f}")

# 3. IC 95%
ic95 = stats.t.interval(0.95, df=len(accuracies)-1,
                        loc=media, scale=stats.sem(accuracies))
print(f"\n¿Supera 0.75?")
print(f"  Límite inferior IC 95%: {ic95[0]:.4f}")
print(f"  {'Sí, con 95% confianza el modelo supera 0.75' if ic95[0] > 0.75 else 'No, el IC incluye valores ≤ 0.75'}")

# 4. n necesario para ancho ≤ 0.05
# Ancho = 2 * z * (std / sqrt(n))  → n = (2 * z * std / ancho)²
z95 = 1.96  # para n grande
ancho_objetivo = 0.05
n_necesario = np.ceil((2 * z95 * std / ancho_objetivo) ** 2)
print(f"\nPara ancho ≤ 0.05: n ≥ {n_necesario:.0f} (tienes {len(accuracies)})")
```
</details>

---
### Ejercicio 2 — A/B test de dos modelos

**Enunciado:** Compara dos versiones de un clasificador:

```python
np.random.seed(1)
f1_v1 = np.random.normal(0.82, 0.04, 25)
f1_v2 = np.random.normal(0.85, 0.06, 25)
```

1. ¿La diferencia de F1 es estadísticamente significativa? (α=0.05)
2. Repite con n=5 y n=200. ¿Cambia la conclusión?
3. ¿Con n=200, sería significativa una diferencia real de solo 0.5%?
4. ¿Qué importa más: el p-valor o el tamaño del efecto?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
from scipy import stats

def test_ab(f1, f2, descripcion):
    t, p = stats.ttest_ind(f1, f2, equal_var=False)
    diff = f2.mean() - f1.mean()
    # Cohen's d (tamaño del efecto)
    s_pool = np.sqrt((f1.std(ddof=1)**2 + f2.std(ddof=1)**2) / 2)
    d = diff / s_pool
    sig = p < 0.05
    print(f"{descripcion:30} diff={diff:+.4f}  p={p:.4f}  d={d:.3f}  {'✓ sig' if sig else '✗ no sig'}")

np.random.seed(1)
# 1 y 2: distintos n
for n in [5, 25, 200]:
    v1 = np.random.normal(0.82, 0.04, n)
    v2 = np.random.normal(0.85, 0.06, n)
    test_ab(v1, v2, f"n={n}")

# 3: diferencia muy pequeña con n=200
np.random.seed(1)
v1_big = np.random.normal(0.820, 0.04, 200)
v2_big = np.random.normal(0.825, 0.04, 200)  # solo 0.5% de diferencia
test_ab(v1_big, v2_big, "n=200, diff=0.005")

print()
print("4. Conclusión:")
print("   Con n grande, diferencias MÍNIMAS son estadísticamente significativas.")
print("   d de Cohen mide el tamaño del efecto: d<0.2 trivial, 0.5 moderado, >0.8 grande.")
print("   Un p-valor significativo no implica que la diferencia sea prácticamente relevante.")
```
</details>

---
### Ejercicio 3 — Muestreo estratificado

**Enunciado:** Tienes un dataset muy desbalanceado:

```python
np.random.seed(3)
clase_0 = np.random.normal(50, 10, 900)   # 90%
clase_1 = np.random.normal(70, 15, 100)   # 10%
```

1. Calcula la media global del dataset completo
2. Toma una muestra aleatoria simple (n=50). ¿Es representativa?
3. Implementa muestreo estratificado (proporcional): 45 de clase_0, 5 de clase_1
4. Compara los ICs de ambos métodos de muestreo vs. la media real

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
from scipy import stats

np.random.seed(3)
clase_0 = np.random.normal(50, 10, 900)
clase_1 = np.random.normal(70, 15, 100)
dataset  = np.concatenate([clase_0, clase_1])

media_real = dataset.mean()
print(f"1. Media real del dataset: {media_real:.4f}")

def ic95(muestra):
    ic = stats.t.interval(0.95, df=len(muestra)-1,
                          loc=muestra.mean(), scale=stats.sem(muestra))
    return muestra.mean(), ic[0], ic[1]

# 2. Muestra aleatoria simple
np.random.seed(42)
muestra_simple = np.random.choice(dataset, size=50, replace=False)
p_clase1_simple = np.sum(muestra_simple > 60) / 50  # proxy de clase_1
m, a, b = ic95(muestra_simple)
print(f"\n2. Muestra simple (n=50):")
print(f"   Media={m:.4f}  IC=[{a:.4f}, {b:.4f}]")
print(f"   ¿Contiene media real? {'Sí' if a <= media_real <= b else 'No'}")

# 3. Muestreo estratificado
m0  = np.random.choice(clase_0, size=45, replace=False)
m1  = np.random.choice(clase_1, size=5,  replace=False)
muestra_estrat = np.concatenate([m0, m1])
m2, a2, b2 = ic95(muestra_estrat)
print(f"\n3. Muestra estratificada (45+5=50):")
print(f"   Media={m2:.4f}  IC=[{a2:.4f}, {b2:.4f}]")
print(f"   ¿Contiene media real? {'Sí' if a2 <= media_real <= b2 else 'No'}")

# 4. Comparativa
print(f"\n4. Media real:            {media_real:.4f}")
print(f"   Error muestra simple:   {abs(m - media_real):.4f}")
print(f"   Error muestra estrat:   {abs(m2 - media_real):.4f}")
print("   → El muestreo estratificado garantiza representación de la clase minoritaria")
```
</details>

---
### Ejercicio 4 — Validación de modelo con bootstrap

**Enunciado:** El bootstrap es una técnica de remuestreo para estimar cualquier estadístico:

```
Para B repeticiones:
  1. Toma una muestra con reemplazo (mismo tamaño que la original)
  2. Calcula el estadístico
El IC bootstrap = percentiles [2.5, 97.5] de las B estimaciones
```

Implementa bootstrap y calcula el IC 95% del F1-score de un clasificador usando:
```python
np.random.seed(7)
predicciones = np.random.binomial(1, 0.8, 200)  # modelo con 80% accuracy aprox
etiquetas    = np.random.binomial(1, 0.5, 200)   # clases balanceadas
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
predicciones = np.random.binomial(1, 0.8, 200)
etiquetas    = np.random.binomial(1, 0.5, 200)

def calcular_f1(pred, real):
    tp = np.sum((pred == 1) & (real == 1))
    fp = np.sum((pred == 1) & (real == 0))
    fn = np.sum((pred == 0) & (real == 1))
    if tp + fp == 0 or tp + fn == 0: return 0.0
    precision = tp / (tp + fp)
    recall    = tp / (tp + fn)
    if precision + recall == 0: return 0.0
    return 2 * precision * recall / (precision + recall)

f1_original = calcular_f1(predicciones, etiquetas)
print(f"F1 en datos originales: {f1_original:.4f}")

# Bootstrap
B = 2000
n = len(predicciones)
f1_boot = np.zeros(B)

for i in range(B):
    idx        = np.random.randint(0, n, size=n)  # con reemplazo
    f1_boot[i] = calcular_f1(predicciones[idx], etiquetas[idx])

ic_inf = np.percentile(f1_boot, 2.5)
ic_sup = np.percentile(f1_boot, 97.5)

print(f"IC bootstrap 95%: [{ic_inf:.4f}, {ic_sup:.4f}]")
print(f"Sesgo estimado:   {f1_boot.mean() - f1_original:.4f}")

plt.figure(figsize=(8, 4))
plt.hist(f1_boot, bins=40, density=True, alpha=0.7, color='steelblue')
plt.axvline(f1_original, color='red',   ls='-',  lw=2, label=f'F1 original={f1_original:.3f}')
plt.axvline(ic_inf,       color='orange', ls='--', lw=2, label=f'IC inferior={ic_inf:.3f}')
plt.axvline(ic_sup,       color='orange', ls='--', lw=2, label=f'IC superior={ic_sup:.3f}')
plt.xlabel('F1-score'); plt.title(f'Bootstrap F1 (B={B})')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 5 (reto) — Motor de A/B testing

**Enunciado:** Implementa una función `ab_test(A, B, alpha=0.05)` que:

1. Realiza t-test, comprueba normalidad (Shapiro) y homogeneidad de varianzas (Levene)
2. Calcula el tamaño del efecto (d de Cohen)
3. Calcula el poder estadístico alcanzado (usando `stats.power`)
4. Devuelve un informe con: ¿diferencia significativa?, magnitud del efecto (trivial/pequeño/moderado/grande), recomendación

Pruébalo con 3 escenarios: sin diferencia, diferencia pequeña (d≈0.2), diferencia grande (d≈0.8).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
from scipy import stats

def ab_test(A, B, alpha=0.05):
    A, B = np.array(A), np.array(B)

    # Normalidad
    _, p_norm_A = stats.shapiro(A)
    _, p_norm_B = stats.shapiro(B)
    normales    = p_norm_A > 0.05 and p_norm_B > 0.05

    # Varianzas
    _, p_levene  = stats.levene(A, B)
    equal_var    = p_levene > 0.05

    # t-test
    t_stat, p_val = stats.ttest_ind(A, B, equal_var=equal_var)
    sig = p_val < alpha

    # Cohen's d
    diff   = B.mean() - A.mean()
    s_pool = np.sqrt((A.std(ddof=1)**2 + B.std(ddof=1)**2) / 2)
    d      = abs(diff / s_pool) if s_pool > 0 else 0

    if   d < 0.2: magnitud = "trivial"
    elif d < 0.5: magnitud = "pequeño"
    elif d < 0.8: magnitud = "moderado"
    else:         magnitud = "grande"

    # Informe
    print("=" * 55)
    print(f"  A: n={len(A)}, μ={A.mean():.4f}, σ={A.std(ddof=1):.4f}")
    print(f"  B: n={len(B)}, μ={B.mean():.4f}, σ={B.std(ddof=1):.4f}")
    print(f"  Diferencia B-A: {diff:+.4f}")
    print(f"  Normalidad:     {'Sí' if normales else 'No (considera test no param.)'}")
    print(f"  Varianzas iguales: {'Sí' if equal_var else 'No (Welch corrección aplicada)'}")
    print(f"  t={t_stat:.4f}  p={p_val:.4f}")
    print(f"  Significativo (α={alpha}): {'Sí ✓' if sig else 'No ✗'}")
    print(f"  d de Cohen: {d:.3f} → efecto {magnitud}")
    print(f"  Recomendación: ", end="")
    if sig and d >= 0.5:
        print("Adoptar B (diferencia significativa y relevante)")
    elif sig and d < 0.2:
        print("Diferencia significativa pero trivial. ¿Merece el coste?")
    elif not sig:
        print("No hay evidencia suficiente. Ampliar muestra o aceptar igualdad.")
    else:
        print("Diferencia moderada. Evaluar coste/beneficio.")
    print()

np.random.seed(0)
ab_test(np.random.normal(5, 1, 50), np.random.normal(5, 1, 50))       # sin diferencia
ab_test(np.random.normal(5, 1, 50), np.random.normal(5.2, 1, 50))     # d≈0.2
ab_test(np.random.normal(5, 1, 50), np.random.normal(5.8, 1, 50))     # d≈0.8
```
</details>

---
## Resumen

```python
from scipy import stats

# Error estándar
se = stats.sem(muestra)               # s / sqrt(n)

# Intervalo de confianza
ic = stats.t.interval(0.95, df=n-1, loc=media, scale=se)

# Tests de hipótesis
t, p = stats.ttest_1samp(x, popmean=val)    # 1 muestra vs valor
t, p = stats.ttest_ind(x, y, equal_var=False)  # 2 muestras independientes
t, p = stats.ttest_rel(x, y)               # pareado (antes/después)

# Diagnóstico
_, p_shapiro = stats.shapiro(x)         # normalidad
_, p_levene  = stats.levene(x, y)       # homogeneidad varianzas
skew         = stats.skew(x)            # asimetría
```

## ¿Qué viene en la próxima clase?
En la **Clase 6** introducimos **pandas**, la herramienta indispensable para análisis de datos reales: Series, DataFrames, filtrado, agrupación y limpieza de datos.